In [ ]:
!pip -q install cmdstanpy pyarrow pandas numpy

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from cmdstanpy import CmdStanModel, install_cmdstan
from pathlib import Path

PARQUET_PATH = Path("/kaggle/input/datasets/karandeepshoker/full-dataset/full_task_all.parquet")
STAN_PATH    = Path("/kaggle/input/datasets/karandeepshoker/bayesian-model/main.stan")
VI_STAN_PATH = Path("/kaggle/input/datasets/karandeepshoker/bayesian-model/VarInf.stan")  # adjust if needed

assert PARQUET_PATH.exists(), PARQUET_PATH
assert STAN_PATH.exists(), STAN_PATH
assert VI_STAN_PATH.exists(), VI_STAN_PATH
print("Found files OK.")

In [ ]:
install_cmdstan(version="2.35.0", cores=2)
print("CmdStan installed.")

In [ ]:
pf = pq.ParquetFile(str(PARQUET_PATH))
md = pf.metadata.metadata or {}

def md_get_str(key: str, default=None):
    b = md.get(key.encode("utf-8"))
    if b is None:
        return default
    try:
        return b.decode("utf-8")
    except Exception:
        return default

K_str = md_get_str("K")
vocab_str = md_get_str("skill_vocab_json")

print("Parquet metadata keys:", sorted([k.decode("utf-8", "ignore") for k in md.keys()])[:20], "...")
print("K metadata:", K_str)
print("Has skill_vocab_json:", vocab_str is not None)

K = int(K_str) if K_str is not None else None
if K is None:
    raise RuntimeError("Missing Parquet metadata 'K'. Your full_task_all.parquet may not be the right build.")

# optional: parse vocab for debugging
skill_vocab = None
if vocab_str:
    # processing/main.ipynb stored `str(dict)` not strict JSON, so parse carefully
    # We'll try JSON first, then eval fallback.
    try:
        skill_vocab = json.loads(vocab_str)
    except Exception:
        try:
            skill_vocab = eval(vocab_str, {"__builtins__": {}})
        except Exception:
            skill_vocab = None

print("K =", K)
print("skill_vocab parsed:", isinstance(skill_vocab, dict))

In [ ]:
df_raw = pd.read_parquet(PARQUET_PATH)

# Expect these columns from processing/main.ipynb
expected = {"QuestionId","UserId","AnswerId","IsCorrect","Gender","TimeTaken","Age","skill_indices"}
missing = expected - set(df_raw.columns)
if missing:
    raise RuntimeError(f"Missing expected columns: {missing}. Columns are: {list(df_raw.columns)[:50]}")

# Filter TimeTaken similarly to main.r
df = df_raw.copy()
df = df[df["TimeTaken"].notna()]
df = df[(df["TimeTaken"] >= 5) & (df["TimeTaken"] <= 1800)].copy()

df["log_time"] = np.log(df["TimeTaken"].astype(float))
df["is_fast"] = (df["TimeTaken"] < 20).astype(np.int32)

print("Rows after filter:", len(df))
df.head()

In [ ]:
# ── Student filtering for HMC feasibility ─────────────────────────────────────
# Apply AFTER TimeTaken filtering.
MIN_ANSWERS_PER_STUDENT = 20
N_STUDENTS = 5000
RANDOM_SEED = 42

counts = df.groupby("UserId").size()
eligible_users = counts[counts >= MIN_ANSWERS_PER_STUDENT].index

print("Eligible students (>=", MIN_ANSWERS_PER_STUDENT, "answers after TimeTaken filter):", len(eligible_users))

if len(eligible_users) < N_STUDENTS:
    raise ValueError(
        f"Only {len(eligible_users)} eligible students, cannot sample N_STUDENTS={N_STUDENTS}. "
        "Lower N_STUDENTS or MIN_ANSWERS_PER_STUDENT."
    )

rng = np.random.default_rng(RANDOM_SEED)
selected_users = rng.choice(eligible_users.to_numpy(), size=N_STUDENTS, replace=False)

# Filter main modeling dataframe to selected students
df = df[df["UserId"].isin(selected_users)].copy()

# Sanity checks
print("Rows after student filter:", len(df))
print("Unique students after filter:", df["UserId"].nunique())

post_counts = df.groupby("UserId").size()
print(
    "Answers per student after filter (min/median/mean/max):",
    int(post_counts.min()),
    float(post_counts.median()),
    float(post_counts.mean()),
    int(post_counts.max()),
)

In [ ]:
# ── Row sampling (after student filter) ─────────────────────────────────────
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

N_TARGET = 50000  # start small on Kaggle; later try 200000 if it runs
ANCHOR_PER_GROUP = 5

# anchors: up to 5 per (UserId, IsCorrect)
anchors = (
    df.groupby(["UserId", "IsCorrect"], group_keys=False)
      .apply(lambda g: g.sample(n=min(len(g), ANCHOR_PER_GROUP), random_state=RANDOM_SEED))
      .reset_index(drop=True)
)

remaining = df[~df["AnswerId"].isin(anchors["AnswerId"])]

need = max(0, N_TARGET - len(anchors))
top_up = remaining.sample(n=min(need, len(remaining)), random_state=RANDOM_SEED)

sample_df = pd.concat([anchors, top_up], ignore_index=True)
sample_df = sample_df.sample(n=min(N_TARGET, len(sample_df)), random_state=RANDOM_SEED).reset_index(drop=True)

print("Sample rows:", len(sample_df))
print("Unique users (should be <= 5000):", sample_df["UserId"].nunique())
print("Unique questions:", sample_df["QuestionId"].nunique())

# hard guard
assert sample_df["UserId"].nunique() <= 5000

In [ ]:
sample_df["player_idx"] = pd.factorize(sample_df["UserId"])[0].astype(np.int32) + 1
sample_df["question_idx"] = pd.factorize(sample_df["QuestionId"])[0].astype(np.int32) + 1

n_players = int(sample_df["player_idx"].max())
n_questions = int(sample_df["question_idx"].max())

assert n_players == sample_df["player_idx"].nunique()
assert n_questions == sample_df["question_idx"].nunique()

print("n_players:", n_players)
print("n_questions:", n_questions)

In [ ]:
# Build mapping from original QuestionId to remapped question_idx
# NOTE: skill_indices entries are list-like (often numpy arrays) and unhashable,
# so we must NOT call drop_duplicates() on [QuestionId, skill_indices].

q_idx_map = (
    sample_df[["QuestionId", "question_idx"]]
    .drop_duplicates(subset=["QuestionId"])
)

q_skill_map = (
    df_raw[["QuestionId", "skill_indices"]]
    .drop_duplicates(subset=["QuestionId"])
)

q_map = (
    q_idx_map
    .merge(q_skill_map, on="QuestionId", how="left")
    .sort_values("question_idx")
    .reset_index(drop=True)
)

assert len(q_map) == n_questions
assert q_map["question_idx"].is_unique

# Ensure each skill_indices is a list[int]
def normalize_skill_list(x):
    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    if isinstance(x, (list, tuple)):
        return [int(v) for v in x]
    try:
        return [int(v) for v in list(x)]
    except Exception:
        return []

skills_per_q = [normalize_skill_list(x) for x in q_map["skill_indices"].tolist()]

# bounds check to [1..K]
skills_per_q = [[s for s in sl if 1 <= s <= K] for sl in skills_per_q]

skill_count = np.array([len(sl) for sl in skills_per_q], dtype=np.int32)
skill_start = np.ones(n_questions, dtype=np.int32)
if n_questions > 1:
    skill_start[1:] = (np.cumsum(skill_count)[:-1] + 1).astype(np.int32)

skill_flat = np.array([s for sl in skills_per_q for s in sl], dtype=np.int32)
n_skill_entries = int(len(skill_flat))

# sanity
assert np.all(skill_count >= 0)
assert np.all(skill_start >= 1)
assert int(skill_count.sum()) == n_skill_entries

print("n_skill_entries:", n_skill_entries)
print("Skill sparsity:", round(1 - n_skill_entries / (n_questions * K), 3))

In [ ]:
player_meta = (
    sample_df[["player_idx","Gender","Age"]]
    .drop_duplicates(subset=["player_idx"])
    .sort_values("player_idx")
    .reset_index(drop=True)
)

assert len(player_meta) == n_players

age_raw = player_meta["Age"].astype(float).to_numpy()
age_mean = np.nanmean(age_raw)
age_sd = np.nanstd(age_raw)
if not np.isfinite(age_sd) or age_sd == 0:
    age_sd = 1.0

age_z = np.where(np.isfinite(age_raw), (age_raw - age_mean) / age_sd, 0.0).astype(np.float64)

gender = player_meta["Gender"].to_numpy()
# match Stan: 0 unknown, 1 ref, 2 other
gender = np.where(pd.isna(gender), 0, gender).astype(np.int32)
gender = np.clip(gender, 0, 2)

print("age_z shape:", age_z.shape)
print("gender counts:", pd.Series(gender).value_counts().to_dict())

In [ ]:
stan_data = {
    "N": int(len(sample_df)),
    "n_players": n_players,
    "n_questions": n_questions,
    "n_skills": int(K),

    "player_id": sample_df["player_idx"].astype(np.int32).to_numpy(),
    "question_id": sample_df["question_idx"].astype(np.int32).to_numpy(),
    "response": sample_df["IsCorrect"].astype(np.int32).to_numpy(),
    "is_fast": sample_df["is_fast"].astype(np.int32).to_numpy(),
    "log_time": sample_df["log_time"].astype(np.float64).to_numpy(),

    "n_skill_entries": int(n_skill_entries),
    "skill_flat": skill_flat.astype(np.int32),
    "skill_start": skill_start.astype(np.int32),
    "skill_count": skill_count.astype(np.int32),

    "age_z": age_z.astype(np.float64),
    "gender": gender.astype(np.int32),
}

# hard checks like main.r
assert stan_data["player_id"].max() == stan_data["n_players"]
assert stan_data["question_id"].max() == stan_data["n_questions"]
assert len(stan_data["age_z"]) == stan_data["n_players"]
assert len(stan_data["gender"]) == stan_data["n_players"]
assert len(stan_data["skill_start"]) == stan_data["n_questions"]
assert len(stan_data["skill_count"]) == stan_data["n_questions"]
print("stan_data assembled OK.")

In [ ]:
# Kaggle input datasets are read-only. CmdStan writes .hpp/exe next to the .stan file,
# so we must copy the Stan files to a writable directory first.
from shutil import copy2

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

STAN_LOCAL_PATH = WORK_DIR / "main.stan"
copy2(STAN_PATH, STAN_LOCAL_PATH)

VI_LOCAL_PATH = None
if VI_STAN_PATH is not None and Path(VI_STAN_PATH).exists():
    VI_LOCAL_PATH = WORK_DIR / Path(VI_STAN_PATH).name
    copy2(VI_STAN_PATH, VI_LOCAL_PATH)

print("Copied Stan files to:")
print(" -", STAN_LOCAL_PATH)
if VI_LOCAL_PATH:
    print(" -", VI_LOCAL_PATH)

model = CmdStanModel(stan_file=str(STAN_LOCAL_PATH))
print("Compiled main.stan OK.")

In [ ]:
fit = model.sample(
    data=stan_data,
    chains=2,
    parallel_chains=2,
    iter_warmup=300,
    iter_sampling=300,
    adapt_delta=0.9,
    max_treedepth=12,
    show_progress=True,
)

print(fit.summary(["mu_skill", "sigma_skill", "gamma_base", "gamma_fast_delta", "alpha", "beta_time", "mu_log_time", "sigma_log_time"]).head(30))

In [ ]:
# Extract a small set of draws for y_rep
draws = fit.draws_pd(vars=["y_rep"])
print("y_rep draws shape:", draws.shape)

obs_mean = float(np.mean(stan_data["response"]))
# draws_pd returns wide columns y_rep[1], y_rep[2], ...
yrep_cols = [c for c in draws.columns if c.startswith("y_rep[")]
pp_mean = float(draws[yrep_cols].to_numpy().mean())

print("Observed mean correct:", round(obs_mean, 4))
print("Posterior predictive mean correct (across draws/obs):", round(pp_mean, 4))

In [ ]:
from cmdstanpy import CmdStanModel

if VI_LOCAL_PATH is None:
    raise FileNotFoundError("VI_STAN_PATH not found (check filename/casing or dataset path).")

vi_model = CmdStanModel(stan_file=str(VI_LOCAL_PATH))
print("Compiled VarInf.stan OK.")

In [ ]:
vi_fit = vi_model.variational(
    data=stan_data,
    algorithm="meanfield",     # or "fullrank" (slower, more flexible)
    iter=20000,
    grad_samples=3,
    elbo_samples=200,
    output_samples=2000,       # posterior draws from the variational approx
)
print("VI done.")
print(vi_fit.variational_params_dict.keys())

In [ ]:
draws = vi_fit.sample  # pandas DataFrame of draws
draws[["gamma_base", "alpha", "beta_time", "mu_log_time", "sigma_log_time"]].describe()

In [ ]:
vi_fit.runset.csv_files  # list of output csv paths